# 从零实现 Masked Autoencoder：只编码可见 patch 的 Tiny MAE

MAE 不是“把像素置零再跑一次完整 ViT”。关键合同是：每个样本独立随机打乱 patch，只把可见 token 送入 encoder；decoder 再插入共享 mask token，并借助 restore index 恢复原空间顺序。这样 encoder 的主计算量从 $O(N^2)$ 降到约 $O(((1-r)N)^2)$，其中 $r$ 是 masking ratio。

本 Notebook 只用 PyTorch 基础层手写 patch 双射、二维位置编码、多头自注意力、逐样本确定性 masking、encoder/decoder 和 masked-only reconstruction loss。内置小数据只验证机制与合同，受控过拟合绝不能解释为 ImageNet 泛化结果。

In [ ]:
import copy
import hashlib
import io
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import torch
from torch import nn
import torch.nn.functional as F

SEED58 = 5801
random.seed(SEED58); torch.manual_seed(SEED58)
torch.set_num_threads(1)
DEVICE58 = torch.device("cpu")

def canonical_json58(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

assert DEVICE58.type == "cpu" and torch.get_num_threads() == 1

## 1. Patch 是可逆的数据布局，不是随意 flatten

对 `images:[B,C,H,W]`，patch 大小为 $P$ 时，输出为 `patches:[B,(H/P)(W/P),P*P*C]`。通道必须在每个 patch 内保持同一约定，否则训练目标虽然 shape 相同，语义却错位。`unpatchify` 显式接收网格尺寸并做逆置换；生产环境还应把颜色空间、resize、归一化和 patch 顺序写入制品契约。

In [ ]:
def patchify58(images, patch_size):
    if images.ndim != 4 or patch_size < 1:
        raise ValueError("images_must_be_BCHW_and_patch_positive")
    b, c, h, w = images.shape
    if h % patch_size or w % patch_size:
        raise ValueError("height_width_must_be_divisible_by_patch")
    gh, gw = h // patch_size, w // patch_size
    x = images.reshape(b, c, gh, patch_size, gw, patch_size)
    return x.permute(0, 2, 4, 3, 5, 1).reshape(b, gh * gw, patch_size * patch_size * c)

def unpatchify58(patches, grid_h, grid_w, channels, patch_size):
    expected = patch_size * patch_size * channels
    if patches.ndim != 3 or patches.shape[1:] != (grid_h * grid_w, expected):
        raise ValueError("patch_shape_does_not_match_grid")
    b = patches.shape[0]
    x = patches.reshape(b, grid_h, grid_w, patch_size, patch_size, channels)
    return x.permute(0, 5, 1, 3, 2, 4).reshape(b, channels, grid_h * patch_size, grid_w * patch_size)

patch_probe58 = torch.arange(2*3*8*12, dtype=torch.float32).reshape(2,3,8,12)
patches_probe58 = patchify58(patch_probe58, 4)
assert patches_probe58.shape == (2,6,48)
assert torch.equal(unpatchify58(patches_probe58,2,3,3,4), patch_probe58)
try:
    patchify58(torch.zeros(1,1,7,8), 4)
    raise AssertionError("non-divisible image should fail")
except ValueError as exc:
    assert "divisible" in str(exc)

## 2. 逐样本确定性 random masking 与 restore index

每个 `sample_id` 派生自己的随机生成器，所以同一样本换 batch 顺序后仍得到相同 mask；这适合回归测试，但正式预训练通常按 epoch 改变 seed。`ids_keep` 指向原始位置，`ids_restore=argsort(ids_shuffle)` 把 `[visible, mask]` 的打乱序列还原。mask 约定为 `0=visible, 1=masked`，不能和 padding mask 混用。

In [ ]:
def random_mask58(tokens, mask_ratio, sample_ids, base_seed=SEED58):
    if tokens.ndim != 3 or not (0.0 < mask_ratio < 1.0):
        raise ValueError("invalid_tokens_or_mask_ratio")
    b, n, d = tokens.shape
    if len(sample_ids) != b or len(set(int(x) for x in sample_ids)) != b:
        raise ValueError("sample_ids_must_be_unique_per_batch")
    keep = max(1, int(n * (1.0 - mask_ratio)))
    shuffles = []
    for sample_id in sample_ids:
        g = torch.Generator(device="cpu").manual_seed(base_seed + 1009 * int(sample_id))
        shuffles.append(torch.rand(n, generator=g).argsort())
    ids_shuffle = torch.stack(shuffles).to(tokens.device)
    ids_restore = ids_shuffle.argsort(1)
    ids_keep = ids_shuffle[:, :keep]
    visible = torch.gather(tokens, 1, ids_keep[...,None].expand(-1,-1,d))
    mask_shuffled = torch.ones(b,n,device=tokens.device)
    mask_shuffled[:,:keep] = 0
    mask = torch.gather(mask_shuffled,1,ids_restore)
    return visible, mask, ids_restore, ids_keep

mask_tokens58 = torch.arange(3*16*2,dtype=torch.float32).reshape(3,16,2)
vis58, mask58, restore58, keep58 = random_mask58(mask_tokens58,0.75,[10,20,30])
assert vis58.shape == (3,4,2) and torch.equal(mask58.sum(1),torch.tensor([12.,12.,12.]))
assert torch.equal(torch.sort(restore58,dim=1).values,torch.arange(16).expand(3,-1))
assert keep58.min()>=0 and keep58.max()<16
manual_shuffle58=torch.tensor([[2,0,3,1]])
manual_restore58=manual_shuffle58.argsort(1)
original_payload58=torch.tensor([[10,11,12,13]])
visible_then_mask58=torch.tensor([[12,10,-1,-1]])  # shuffle 前两项可见，其余插入 mask token
manual_recovered58=torch.gather(visible_then_mask58,1,manual_restore58)
assert torch.equal(manual_recovered58,torch.tensor([[10,-1,12,-1]]))
assert not torch.equal(torch.gather(visible_then_mask58,1,manual_shuffle58),manual_recovered58)  # 错把 shuffle 当 restore 必须失败
assert torch.equal(keep58,restore58.argsort(1)[:,:keep58.shape[1]])
_, reordered_mask58, _, _ = random_mask58(mask_tokens58[[2,0,1]],.75,[30,10,20])
assert torch.equal(reordered_mask58,mask58[[2,0,1]])

## 3. 固定二维位置编码与手写 attention

patch 被抽样后仍须携带原位置。这里把行、列分别编码到一半通道，要求维度能被 4 整除。注意力显式计算 $\mathrm{softmax}(QK^T/\sqrt{d_h})V$，没有调用 `nn.MultiheadAttention` 或现成 Transformer。shape 从 `[B,N,D]` 拆为 `[B,h,N,d_h]`；encoder 的 $N$ 仅为可见 patch 数。

In [ ]:
def sincos_2d58(grid_h, grid_w, dim):
    if dim % 4 or grid_h < 1 or grid_w < 1:
        raise ValueError("position_dim_must_be_multiple_of_four")
    y, x = torch.meshgrid(torch.arange(grid_h,dtype=torch.float32),torch.arange(grid_w,dtype=torch.float32),indexing="ij")
    omega = 1.0 / (10000 ** (torch.arange(dim//4,dtype=torch.float32)/(dim//4)))
    ex, ey = x.reshape(-1,1)*omega, y.reshape(-1,1)*omega
    return torch.cat([ex.sin(),ex.cos(),ey.sin(),ey.cos()],1)

class ManualSelfAttention58(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        if dim % heads: raise ValueError("dim_not_divisible_by_heads")
        self.dim, self.heads, self.head_dim = dim, heads, dim//heads
        self.qkv = nn.Linear(dim,3*dim); self.out = nn.Linear(dim,dim)
    def forward(self,x):
        if x.ndim != 3 or x.shape[-1] != self.dim: raise ValueError("attention_input_shape")
        b,n,_ = x.shape
        q,k,v = self.qkv(x).reshape(b,n,3,self.heads,self.head_dim).permute(2,0,3,1,4)
        weights = (q @ k.transpose(-2,-1) / math.sqrt(self.head_dim)).softmax(-1)
        return self.out((weights @ v).transpose(1,2).reshape(b,n,self.dim)), weights

class EncoderBlock58(nn.Module):
    def __init__(self,dim,heads=4):
        super().__init__(); self.n1=nn.LayerNorm(dim); self.attn=ManualSelfAttention58(dim,heads); self.n2=nn.LayerNorm(dim)
        self.ff=nn.Sequential(nn.Linear(dim,3*dim),nn.GELU(),nn.Linear(3*dim,dim))
    def forward(self,x):
        a,w=self.attn(self.n1(x)); x=x+a; return x+self.ff(self.n2(x)),w

pos58=sincos_2d58(4,4,32)
attn_out58,attn_w58=ManualSelfAttention58(32)(torch.randn(2,5,32))
assert pos58.shape==(16,32) and attn_out58.shape==(2,5,32)
assert torch.allclose(attn_w58.sum(-1),torch.ones(2,4,5),atol=1e-6)

## 4. Tiny MAE：encoder 只看 visible token，decoder 恢复完整序列

patch projection 后先加 encoder 位置编码，再 gather 可见 token。decoder 将可见 latent 投影到较小维度，与共享 mask token 拼成 shuffle 顺序，经 `ids_restore` 回到空间顺序，最后加 decoder 位置编码并预测每个 patch。decoder 轻量是 MAE 的计算设计，不是信息泄漏：masked 像素从未进入 encoder。

In [ ]:
class TinyMAE58(nn.Module):
    def __init__(self,image_size=8,patch_size=2,in_chans=1,enc_dim=32,dec_dim=24,heads=4):
        super().__init__()
        if image_size%patch_size or enc_dim%4 or dec_dim%4: raise ValueError("invalid_mae_config")
        self.image_size,self.patch_size,self.in_chans=image_size,patch_size,in_chans
        self.grid=image_size//patch_size; self.patch_dim=patch_size*patch_size*in_chans
        self.patch_embed=nn.Linear(self.patch_dim,enc_dim)
        self.register_buffer("enc_pos",sincos_2d58(self.grid,self.grid,enc_dim))
        self.encoder=EncoderBlock58(enc_dim,heads)
        self.enc_to_dec=nn.Linear(enc_dim,dec_dim); self.mask_token=nn.Parameter(torch.zeros(1,1,dec_dim))
        self.register_buffer("dec_pos",sincos_2d58(self.grid,self.grid,dec_dim))
        self.decoder=EncoderBlock58(dec_dim,heads); self.pred=nn.Linear(dec_dim,self.patch_dim)
        nn.init.normal_(self.mask_token,std=.02)
    def forward(self,images,sample_ids,mask_ratio=.75):
        if images.shape[1:]!=(self.in_chans,self.image_size,self.image_size): raise ValueError("image_contract_mismatch")
        patches=patchify58(images,self.patch_size)
        embedded=self.patch_embed(patches)+self.enc_pos[None]
        visible,mask,ids_restore,ids_keep=random_mask58(embedded,mask_ratio,sample_ids)
        latent,_=self.encoder(visible); latent=self.enc_to_dec(latent)
        mask_tokens=self.mask_token.expand(images.shape[0],patches.shape[1]-latent.shape[1],-1)
        restored=torch.gather(torch.cat([latent,mask_tokens],1),1,ids_restore[...,None].expand(-1,-1,latent.shape[-1]))
        decoded,_=self.decoder(restored+self.dec_pos[None])
        return self.pred(decoded),mask,ids_restore,ids_keep

def masked_patch_loss58(pred,target,mask):
    if pred.shape!=target.shape or mask.shape!=pred.shape[:2] or mask.sum()<=0: raise ValueError("invalid_masked_loss_inputs")
    per_patch=(pred-target).pow(2).mean(-1)
    return (per_patch*mask).sum()/mask.sum()

mae_probe58=TinyMAE58(); probe_images58=torch.randn(3,1,8,8)
pred58,model_mask58,model_restore58,model_keep58=mae_probe58(probe_images58,[1,2,3])
assert pred58.shape==(3,16,4) and model_keep58.shape==(3,4)
assert torch.equal(model_restore58.argsort(1).argsort(1),model_restore58)

## 5. 关键 oracle：loss 范围、信息路径与梯度

仅比较最终 loss 下降无法发现“把全部 patch 偷送进 encoder”。下面先证明 visible 位置预测怎么改都不影响 masked loss；再对输入求梯度：masked target 被 `detach` 后，masked 输入像素梯度必须为零，而可见输入必须通过 attention 影响重建。最后干预可见 patch，masked prediction 应发生变化，证明 decoder 不是只输出常数模板。

In [ ]:
target58=patchify58(probe_images58,2)
base_loss58=masked_patch_loss58(pred58,target58,model_mask58)
visible_changed58=pred58.detach().clone(); visible_changed58[model_mask58==0]+=999
masked_changed58=pred58.detach().clone(); masked_changed58[model_mask58==1]+=1
assert torch.allclose(masked_patch_loss58(visible_changed58,target58,model_mask58),base_loss58.detach())
assert masked_patch_loss58(masked_changed58,target58,model_mask58)>base_loss58.detach()+.5

grad_image58=torch.randn(1,1,8,8,requires_grad=True)
grad_pred58,grad_mask58,_,_=mae_probe58(grad_image58,[77])
masked_patch_loss58(grad_pred58,patchify58(grad_image58,2).detach(),grad_mask58).backward()
grad_by_patch58=patchify58(grad_image58.grad,2).abs().sum(-1)
assert grad_by_patch58[grad_mask58==0].sum()>0
assert grad_by_patch58[grad_mask58==1].max()<1e-10

with torch.no_grad():
    original_pred58,mi58,_,_=mae_probe58(probe_images58[:1],[91])
    altered_patches58=patchify58(probe_images58[:1],2).clone(); altered_patches58[mi58==0]+=2
    altered_image58=unpatchify58(altered_patches58,4,4,1,2)
    altered_pred58,_,_,_=mae_probe58(altered_image58,[91])
assert (original_pred58[mi58==1]-altered_pred58[mi58==1]).abs().mean()>1e-4

## 6. 受控训练：低维图案重建

合成图像由全局强度、横向坡度与纵向坡度生成，因此可见 patch 包含推断被遮区域的信息。固定 sample-id mask 让回归稳定；真实 MAE 应随 epoch 改变 mask、用更大 encoder、数据增强和独立验证集。本实验只要求训练 loss 明显下降，并报告 held-out 受控图案误差，不宣称自然图像能力。

In [ ]:
def make_images58(count=48):
    g=torch.Generator().manual_seed(SEED58+2)
    yy,xx=torch.meshgrid(torch.linspace(-1,1,8),torch.linspace(-1,1,8),indexing="ij")
    coeff=torch.rand(count,3,generator=g)*1.4-.7
    images=coeff[:,0,None,None]+coeff[:,1,None,None]*xx+coeff[:,2,None,None]*yy
    return images[:,None].float()

all_images58=make_images58(); split58={"train":list(range(40)),"val":list(range(40,48))}
train_images58=all_images58[:40]; train_ids58=list(range(40))
torch.manual_seed(SEED58+3); model58=TinyMAE58()
optimizer58=torch.optim.AdamW(model58.parameters(),lr=8e-3,weight_decay=1e-4)
losses58=[]
for step58 in range(61):
    optimizer58.zero_grad()
    out58,m58,_,_=model58(train_images58,train_ids58,.75)
    loss58=masked_patch_loss58(out58,patchify58(train_images58,2),m58)
    loss58.backward(); optimizer58.step(); losses58.append(float(loss58.detach()))
model58.eval()
with torch.no_grad():
    val_out58,val_mask58,_,_=model58(all_images58[40:],split58["val"],.75)
    val_loss58=float(masked_patch_loss58(val_out58,patchify58(all_images58[40:],2),val_mask58))
assert losses58[-1] < losses58[0]*.25 and val_loss58 < .12
assert all(math.isfinite(x) for x in losses58) and math.isfinite(val_loss58)
print({"initial":round(losses58[0],4),"final":round(losses58[-1],4),"controlled_val":round(val_loss58,4)})

## 7. 推理重建与边界

可视化时通常保留 visible patch，并只用预测替换 masked patch；直接展示全部预测会混淆 decoder 的训练目标。确定性 `sample_id` 也是请求协议的一部分。对真实服务还要限制图像尺寸、拒绝 NaN/Inf、记录 mask seed，并区分预训练重建和下游 fine-tune 推理。

In [ ]:
@torch.no_grad()
def reconstruct58(model,images,sample_ids,mask_ratio=.75):
    if not torch.isfinite(images).all(): raise ValueError("nonfinite_image")
    pred,mask,_,_=model(images,sample_ids,mask_ratio)
    original=patchify58(images,model.patch_size)
    merged=torch.where(mask[...,None].bool(),pred,original)
    return unpatchify58(merged,model.grid,model.grid,model.in_chans,model.patch_size),mask

reconstruction58,reconstruction_mask58=reconstruct58(model58,all_images58[40:42],[40,41])
visible_original58=patchify58(all_images58[40:42],2)[reconstruction_mask58==0]
visible_rebuilt58=patchify58(reconstruction58,2)[reconstruction_mask58==0]
assert reconstruction58.shape==(2,1,8,8)
assert torch.equal(visible_original58,visible_rebuilt58)
try:
    reconstruct58(model58,torch.full((1,1,8,8),float("nan")),[1])
    raise AssertionError("NaN should fail")
except ValueError as exc: assert "nonfinite" in str(exc)

## 8. 发布制品：哈希不是信任根

`state_digest` 逐参数绑定 key、dtype、shape 与原始 bytes；manifest 绑定训练数据、split、预处理和训练 recipe。但调用者若能同时替换 payload 和其中的哈希，普通“自校验”仍可伪造。因此 loader 只信任包外、只读 publisher registry 中预先发布的整体摘要，并返回限制了输入协议的 `PublishedMAE` wrapper。

In [ ]:
def tensor_digest58(tensor):
    t=tensor.detach().cpu().contiguous(); h=hashlib.sha256()
    h.update(str(t.dtype).encode()); h.update(canonical_json58(list(t.shape)).encode()); h.update(t.numpy().tobytes())
    return h.hexdigest()

def state_digest58(state):
    h=hashlib.sha256()
    for key in sorted(state):
        t=state[key].detach().cpu().contiguous()
        h.update(key.encode()); h.update(str(t.dtype).encode()); h.update(canonical_json58(list(t.shape)).encode()); h.update(t.numpy().tobytes())
    return h.hexdigest()

def artifact_digest58(package):
    signed={"subject":package["subject"],"manifest":package["manifest"],"state_digest":state_digest58(package["state"])}
    return hashlib.sha256(canonical_json58(signed).encode()).hexdigest()

config58={"image_size":8,"patch_size":2,"in_chans":1,"enc_dim":32,"dec_dim":24,"heads":4}
manifest58={
    "config":config58,
    "data":{"train_tensor":tensor_digest58(train_images58),"all_tensor":tensor_digest58(all_images58)},
    "split":split58,
    "preprocess":{"color":"single-channel-linear","resize":"none","range":"unbounded-synthetic","patch_order":"row-major-HWPC"},
    "recipe":{"seed":SEED58+3,"optimizer":"AdamW","steps":61,"lr":8e-3,"mask_ratio":.75,"mask_seed":SEED58},
}
package58={"subject":"tiny-mae58/v1","manifest":copy.deepcopy(manifest58),"state":copy.deepcopy(model58.state_dict())}
package58["artifact_digest"]=artifact_digest58(package58)
_PUBLISHER_REGISTRY58=MappingProxyType({package58["subject"]:package58["artifact_digest"]})

def deep_freeze58(value):
    if isinstance(value,dict): return MappingProxyType({key:deep_freeze58(item) for key,item in value.items()})
    if isinstance(value,(list,tuple)): return tuple(deep_freeze58(item) for item in value)
    return value

class PublishedMAE58:
    def __init__(self,model,manifest): self._model=model.eval(); self.manifest=deep_freeze58(copy.deepcopy(manifest))
    def reconstruct(self,images,sample_ids):
        if images.shape[1:]!=(1,8,8): raise ValueError("published_input_contract")
        return reconstruct58(self._model,images,sample_ids,self.manifest["recipe"]["mask_ratio"])

def load_published58(package):
    actual=artifact_digest58(package)
    if _PUBLISHER_REGISTRY58.get(package.get("subject"))!=actual: raise ValueError("publisher_digest_mismatch")
    if package.get("artifact_digest")!=actual or package["manifest"]!=manifest58: raise ValueError("manifest_or_embedded_digest_mismatch")
    loaded=TinyMAE58(**package["manifest"]["config"]); loaded.load_state_dict(package["state"],strict=True)
    return PublishedMAE58(loaded,package["manifest"])

published58=load_published58(package58)
pub_image58,_=published58.reconstruct(all_images58[40:41],[40])
assert pub_image58.shape==(1,1,8,8) and isinstance(published58.manifest,MappingProxyType)
published_ratio58=published58.manifest["recipe"]["mask_ratio"]
try:
    published58.manifest["recipe"]["mask_ratio"]=.1
    raise AssertionError("nested published recipe was mutable")
except TypeError: pass
assert published58.manifest["recipe"]["mask_ratio"]==published_ratio58==.75
forged58=copy.deepcopy(package58); first_key58=next(iter(forged58["state"])); forged58["state"][first_key58].zero_()
forged58["manifest"]["recipe"]["steps"]=1
forged58["artifact_digest"]=artifact_digest58(forged58)  # 攻击者整体重签，仍不在 publisher registry
try:
    load_published58(forged58); raise AssertionError("re-signed forgery should fail")
except ValueError as exc: assert "publisher" in str(exc)

## 9. 复杂度、失败模式与生产差距

- patchify 为 $O(BHWC)$；encoder attention 为 $O(B((1-r)N)^2D)$；decoder 仍为 $O(BN^2D_d)$，所以应让 decoder 浅且窄。
- 常见错误包括 restore index 方向反了、mask 语义颠倒、对所有 patch 求 loss、不同样本共用同一随机排列，以及验证集沿用训练数据统计却未版本化。
- 本实现没有多层大模型、mixed precision、分布式训练、随机增强、梯度累积和下游 fine-tune；合成图案存在强低维结构，低 loss 只是单元测试。
- 生产制品还需要真实签名/KMS、不可变对象存储、依赖 SBOM、审批与回滚；进程内 `MappingProxyType` 只演示“信任根必须在包外”。

## 10. 原始资料

- He et al., [Masked Autoencoders Are Scalable Vision Learners](https://arxiv.org/abs/2111.06377)
- Dosovitskiy et al., [An Image is Worth 16x16 Words](https://arxiv.org/abs/2010.11929)
- PyTorch 官方文档：[Reproducibility](https://pytorch.org/docs/stable/notes/randomness.html)

论文给出方法与实验尺度；本 Notebook 刻意缩成 CPU 可运行的机制验证，结果不可横向对比论文指标。